**Location Mapping**

In [0]:
gold_location = "abfss://gold@walmartpstorageaccount.dfs.core.windows.net/"

In [0]:
parameters = [
    {
        "table": 'customers',
        "name":'dim_customers',
        "key": "customer_id",
        "timestamp": "customers_updated_timestamp",
        "columns":'''
                    customer_id,
                    customers_first_name,
                    customers_last_name,
                    customers_email,
                    customers_phone,
                    customers_city,
                    customers_province,
                    customers_country,
                    customers_created_timestamp,
                    customers_updated_timestamp,
                    customers_is_active,
                    customers_processed_timestamp,
                    current_timestamp() AS customers_gold_loaded_timestamp
                  ''',
        "rank": 'DENSE_RANK() OVER(PARTITION BY customer_id ORDER BY customers_updated_timestamp DESC) AS `rank`',
        "condition": 'WHERE customers_updated_timestamp > (SELECT MAX(customers_updated_timestamp) FROM walmart.gold.dim_customers)',
        "location" : gold_location
    },
    {
        "table": 'employees',
        "name":'dim_employees',
        "key": "employee_id",
        "timestamp": "employees_updated_timestamp",
        "columns":'''
                    employee_id,
                    employees_first_name,
                    employees_last_name,
                    employees_email,
                    job_title,
                    salary,
                    store_id,
                    employees_created_timestamp,
                    employees_updated_timestamp,
                    employees_is_active,
                    employees_processed_timestamp,
                    current_timestamp() AS employees_gold_loaded_timestamp
                  ''',
        "rank": 'DENSE_RANK() OVER(PARTITION BY employee_id ORDER BY employees_updated_timestamp DESC) AS `rank`',
        "condition": 'WHERE employees_updated_timestamp > (SELECT MAX(employees_updated_timestamp) FROM walmart.gold.dim_employees)',
        "location" : gold_location
    },
    {
        "table": 'orders',
        "name":'dim_orders',
        "key": "order_item_id",
        "timestamp": "orders_updated_timestamp",
        "columns":'''
                    order_id,
                    order_item_id,
                    payment_method,
                    order_status,
                    order_timestamp,
                    orders_created_timestamp,
                    orders_updated_timestamp,
                    orders_is_active,
                    orders_processed_timestamp,
                    obt_processed_timestamp,
                    current_timestamp() AS orders_gold_loaded_timestamp
                  ''',
        "rank": 'DENSE_RANK() OVER(PARTITION BY order_id ORDER BY orders_updated_timestamp DESC) AS `rank`',
        "condition": 'WHERE orders_updated_timestamp > (SELECT MAX(orders_updated_timestamp) FROM walmart.gold.dim_orders)',
        "location" : gold_location
    },
    {
        "table": 'products',
        "name":'dim_products',
        "key": "product_id",
        "timestamp": "products_updated_timestamp",
        "columns":'''
                    product_id,
                    product_name,
                    category,
                    brand,
                    price,
                    products_created_timestamp,
                    products_updated_timestamp,
                    products_is_active,
                    products_processed_timestamp,
                    current_timestamp() AS products_gold_loaded_timestamp
                  ''',
        "rank": 'DENSE_RANK() OVER(PARTITION BY product_id ORDER BY products_updated_timestamp DESC) AS `rank`',
        "condition": 'WHERE products_updated_timestamp > (SELECT MAX(products_updated_timestamp) FROM walmart.gold.dim_products)',
        "location" : gold_location
    },
    {
        "table": 'stores',
        "name":'dim_stores',
        "key": "store_id",
        "timestamp": "stores_updated_timestamp",
        "columns":'''
                    store_id,
                    store_name,
                    stores_city,
                    stores_province,
                    stores_country,
                    stores_created_timestamp,
                    stores_updated_timestamp,
                    stores_is_active,
                    stores_processed_timestamp,
                    current_timestamp() AS stores_gold_loaded_timestamp
                  ''',
        "rank": 'DENSE_RANK() OVER(PARTITION BY store_id ORDER BY stores_updated_timestamp DESC) AS `rank`',
        "condition": 'WHERE stores_updated_timestamp > (SELECT MAX(stores_updated_timestamp) FROM walmart.gold.dim_stores)',
        "location" : gold_location
    },
    {
        "name":'fact_orders',
        "columns":'''
                    order_id,
                    order_item_id,
                    product_id,
                    store_id,
                    employee_id,
                    customer_id,
                    total_amount,
                    quantity,
                    unit_price,
                    line_amount
                  ''',
        "location" : gold_location
    },
]

In [0]:
from jinja2 import Template

jinja_creation_query_dim = """
                            CREATE TABLE IF NOT EXISTS walmart.gold.{{param['name']}}
                            USING DELTA
                            LOCATION '{{param['location']}}{{param['name']}}'
                            AS
                            SELECT
                                * EXCEPT (rank),
                                CASE
                                    WHEN rank = 1 THEN 'Updated'
                                    ELSE 'Historical'
                                END AS `{{param['table']}}_details_status`
                            FROM(SELECT
                                    *,
                                    {{param['rank']}}
                                FROM(SELECT DISTINCT
                                        {{param['columns']}}
                                    FROM walmart.silver.obt_silver)t)
                        """
jinja_new_versions_query_dim = """
                                CREATE OR REPLACE TEMP VIEW {{param['name']}}_view AS

                                WITH cte_1 AS (
                                    SELECT DISTINCT
                                        {{param['columns']}}
                                    FROM walmart.silver.obt_silver
                                    WHERE {{param['timestamp']}} IS NOT NULL
                                ),

                                cte_2 AS (
                                    SELECT
                                        {{param['key']}},
                                        MAX({{param['timestamp']}}) AS max_updated_timestamp
                                    FROM walmart.gold.{{param['name']}}
                                    GROUP BY {{param['key']}}
                                ),

                                cte_3 AS (
                                    SELECT s.*
                                    FROM cte_1 AS s
                                    LEFT JOIN cte_2 AS t
                                        ON s.{{param['key']}} = t.{{param['key']}}
                                    WHERE t.{{param['key']}} IS NULL
                                    OR t.max_updated_timestamp IS NULL
                                    OR s.{{param['timestamp']}} > t.max_updated_timestamp
                                ),

                                cte_4 AS (
                                    SELECT
                                        *,
                                        DENSE_RANK() OVER (
                                            PARTITION BY {{param['key']}}
                                            ORDER BY {{param['timestamp']}} DESC
                                        ) AS version_rank
                                    FROM cte_3
                                )

                                SELECT
                                    * EXCEPT (version_rank)
                                FROM cte_4
                                WHERE version_rank = 1
                            """

jinja_expire_query_dim = """
                            MERGE INTO walmart.gold.{{param['name']}} AS t
                            USING (
                                SELECT
                                    {{param['key']}},
                                    {{param['timestamp']}}
                                FROM {{param['name']}}_view
                            ) AS s
                            ON t.{{param['key']}} = s.{{param['key']}}
                            AND t.{{param['table']}}_details_status = 'Updated'

                            WHEN MATCHED
                                AND s.{{param['timestamp']}} > t.{{param['timestamp']}}
                            THEN UPDATE SET
                                {{param['table']}}_details_status = 'Historical'
                        """

jinja_insert_query_dim = """
                            INSERT INTO TABLE walmart.gold.{{param['name']}}
                            SELECT
                                *,
                                'Updated' AS {{param['table']}}_details_status
                            FROM {{param['name']}}_view
                        """

jinja_fact_query = """
                    SELECT
                        {{param['columns']}}
                    FROM walmart.silver.obt_silver
                   """